In [6]:
import os
import pandas as pd

# Try multiple possible locations
possible_paths = [
    'insurance_data.csv',          # root directory
    'data/insurance_data.csv',     # inside data/ folder
    '../insurance_data.csv',       # one level up (if notebook is in a subfolder)
    '../data/insurance_data.csv',  # one level up then data/
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Data loaded from: {path}")
        break

if df is None:
    raise FileNotFoundError("Could not find insurance_data.csv in any expected location.")

✅ Data loaded from: insurance_data.csv


In [7]:
# %% [markdown]
# # Task 3 – Hypothesis Testing

# %%
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
import os

# Auto-detect CSV file
possible_paths = ['insurance_data.csv', 'data/insurance_data.csv', '../insurance_data.csv', '../data/insurance_data.csv']
df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Data loaded from: {path}")
        break
if df is None:
    raise FileNotFoundError("Could not find insurance_data.csv")

# Create derived columns
df['ClaimStatus'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']
df['LossRatio'] = df['LossRatio'].replace([np.inf, -np.inf], np.nan).fillna(0)

# Helper functions
def test_two_proportions(df, group_col, group_a, group_b, target='ClaimStatus'):
    a = df[df[group_col] == group_a]
    b = df[df[group_col] == group_b]
    count = [a[target].sum(), b[target].sum()]
    nobs = [len(a), len(b)]
    z, p = proportions_ztest(count, nobs)
    return f"{group_col}: {group_a} vs {group_b}: p={p:.4f} (reject H0 if p<0.05: {p<0.05})"

def test_anova(df, group_col, target='LossRatio'):
    groups = [df[df[group_col] == g][target].dropna() for g in df[group_col].unique()]
    f, p = stats.f_oneway(*groups)
    return f"{group_col} differences in {target}: p={p:.4f} (reject H0 if p<0.05: {p<0.05})"

def test_t_test(df, group_col, group_a, group_b, target='Margin'):
    a = df[df[group_col] == group_a][target]
    b = df[df[group_col] == group_b][target]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    return f"{group_col}: {group_a} vs {group_b} ({target}): p={p:.4f} (reject H0 if p<0.05: {p<0.05})"

# Run tests
print("=== HYPOTHESIS TESTING RESULTS ===\n")
print(test_anova(df, 'Province', 'LossRatio'))

top_zips = df['ZipCode'].value_counts().head(2).index.tolist()
if len(top_zips) >= 2:
    print(test_two_proportions(df, 'ZipCode', top_zips[0], top_zips[1]))
    print(test_t_test(df, 'ZipCode', top_zips[0], top_zips[1], 'Margin'))
else:
    print("Zip code tests skipped – not enough distinct codes.")

genders = df['Gender'].unique()
if len(genders) >= 2:
    print(test_two_proportions(df, 'Gender', genders[0], genders[1]))
else:
    print("Gender test skipped.")

✅ Data loaded from: insurance_data.csv
=== HYPOTHESIS TESTING RESULTS ===

Province differences in LossRatio: p=0.0928 (reject H0 if p<0.05: False)
ZipCode: 10004 vs 10002: p=0.1968 (reject H0 if p<0.05: False)
ZipCode: 10004 vs 10002 (Margin): p=0.2642 (reject H0 if p<0.05: False)
Gender: Male vs Female: p=0.9417 (reject H0 if p<0.05: False)
